# Setting Initial Conditions

An alternative to including a warm-up period in a model is to set initial conditions.  In our stroke example this would mean adding patients to beds and if required the admission queue.  

🎓 **Why use Initial-Conditions over a Warm-Up?**

* While warm-up periods are simpler to code, they can also increase how long it takes to run your model.  Setting initial conditions is in theory computationally more efficient (no wasted run-time).

* Setting initial-conditions also allows you to test specific starting scenarios, such as "What happens if the day begins with a full ward?"

**In general, adding initial conditions is more complex than including a warm-up period.**

Initial conditions could be 

1. Fixed i.e. the same number of patients for each replication.
2. Random i.e. sampled from a discrete distribution specified by a user.

In this notebook we will learn how to add initial conditions to a SimPy model.  We will again use the acute stroke pathway model from the warm-up example notebook.  Our initial conditions approach will load stroke patients to the queue before the simulation begins.

<img src="img/acute_stroke_pathway.png" alt="stroke pathway" width="400"/>


## IMPORTANT: There is still an initialisation bias problem

* We load patients into the model before we begin the run. 
* If there are $n$ acute stroke beds then the first $n$ patients loaded into the queue will begin service immediately and have a zero queuing time.
* This applies if we have one queue in the model or if we have multiple queues and activities: the time in system metrics will be biased for initial condition patients.
* This is the same problem faced when starting the model from time zero with no patients.

**Some Options:**
1. Include a setting in a process to switch results collection on and off.
2. Code a seperate process for initial conditions that does not include results collection.
3. Mixed initial conditions and (a shorter) warm-up period.
    * You will need to do some analysis to ensure this is working acceptably.
    * The warm-up pushes patients into service and also resets results collection. (deletes an initial transient).

## 1. Imports 

In [1]:
import numpy as np
import itertools
import simpy

In [2]:
# to reduce code these classes can be found in distribution.py
from distributions import (
    Exponential, 
    Lognormal, 
    DiscreteEmpirical,
    FixedDistribution
)

## 2. Constants

In [3]:
# default mean inter-arrival times(exp)
# time is in days
IAT_STROKES = 1.0

# resources
N_ACUTE_BEDS = 9

# Acute LoS (Lognormal)
ACUTE_LOS_MEAN = 7.0
ACUTE_LOS_STD = 1.0

# initial conditions for acute queue + service 
# if there are 9 beds then 10 = 1 queuing
# < 9 = 0 queuing etc.
# these can be fixed or random
# Note we are adding N_ACUTE_BEDS patients to queue lengths

INIT_COND_PARAMS = {
    "mode": "fixed",
    "fixed": 10,
    "rnd": {
        "values":[8, 9, 10, 11, 12, 13],
        "freq":[15, 15, 25, 20, 5, 5]
    },
    "collect_results": False
}
    
# sampling settings
N_STREAMS = 3
DEFAULT_RND_SET = 0

# Boolean switch to display simulation results as the model runs
TRACE = False

# run variables (units = days)
WU_PERIOD = 0.0
RC_PERIOD = 100

## 2. Helper classes and functions

In [4]:
def trace(msg):
    """
    Turning printing of events on and off.

    Params:
    -------
    msg: str
        string to print to screen.
    """
    if TRACE:
        print(msg)

## 3. Experiment class

In [5]:
class Experiment:
    """
    Encapsulates the concept of an experiment 🧪 for the stroke pathway
    bed blocking simulator. Manages parameters, PRNG streams and results.

    There is a new parameter `init_cond_params` that stores the initial
    conditions to use in an experiment.
    """
    def __init__(
        self,
        random_number_set=DEFAULT_RND_SET,
        n_streams=N_STREAMS,
        iat_strokes=IAT_STROKES,
        acute_los_mean=ACUTE_LOS_MEAN,
        acute_los_std=ACUTE_LOS_STD,
        n_acute_beds=N_ACUTE_BEDS,
        init_cond_params=INIT_COND_PARAMS,  
    ):
        """
        The init method sets up our defaults.
        """
        # sampling
        self.random_number_set = random_number_set
        self.n_streams = n_streams

        # store parameters for the run of the model
        self.iat_strokes = iat_strokes
        self.acute_los_mean = acute_los_mean
        self.acute_los_std = acute_los_std

        # ---- stored initial conditions -------
        self.init_cond_params = init_cond_params

        #  place holder for resources
        self.acute_ward = None
        self.n_acute_beds = n_acute_beds
        
        # initialise results to zero
        self.init_results_variables()

        # initialise sampling objects
        self.init_sampling()

    def set_random_no_set(self, random_number_set):
        """
        Controls the random sampling
        
        Parameters:
        -----------
        random_number_set: int
            Used to control the set of pseudo random numbers used by
            the distributions in the simulation.
        """
        self.random_number_set = random_number_set
        self.init_sampling()

    def init_sampling(self):
        """
        Create the distributions used by the model and initialise
        the random seeds of each.
        """
        # produce n non-overlapping streams
        seed_sequence = np.random.SeedSequence(self.random_number_set)
        self.seeds = seed_sequence.spawn(self.n_streams)

        # create distributions

        # inter-arrival time distributions
        self.arrival_strokes = Exponential(
            self.iat_strokes, random_seed=self.seeds[0]
        )

        self.acute_los = Lognormal(
            self.acute_los_mean, self.acute_los_std, random_seed=self.seeds[1]
        )

        if self.init_cond_params["mode"] == "fixed":
            self.init_conds = FixedDistribution(
                self.init_cond_params["fixed"]
            )
        elif self.init_cond_params["mode"] == "rnd":
            self.init_conds = DiscreteEmpirical(
                values = self.init_cond_params["rnd"]["values"],
                freq = self.init_cond_params["rnd"]["freq"],
                random_seed=self.seeds[2]
            )
        else:
            raise ValueError("Initial conditions mode must be 'fixed' or 'rnd'")
            

    def init_results_variables(self):
        """
        Initialise all of the experiment variables used in results
        collection.  This method is called at the start of each run
        of the model
        """
        # variable used to store results of experiment
        self.results = {}
        self.results["n_arrivals"] = 0
        self.results["waiting_acute"] = []

## 🥵 Warm-up period

In [6]:
def warmup_complete(warm_up_period, env, args):
    """
    End of warm-up period event. Used to reset results collection variables.

    Parameters:
    ----------
    warm_up_period: float
        Duration of warm-up period in simulation time units

    env: simpy.Environment
        The simpy environment

    args: Experiment
        The simulation experiment that contains the results being collected.
    """
    yield env.timeout(warm_up_period)
    trace(f"{env.now:.2f}: 🥵 Warm up complete.")
    
    args.init_results_variables()

## 4. Pathway process logic

The key things to recognise are 

* We include a optional parameter called `collect_results` that defaults to `True`. We may set this `False` in our functions that setup initial conditions

In [7]:
def acute_stroke_pathway(patient_id, env, args, collect_results=True):
    """Process a patient through the acute ward. Simpy generator function.
    
    Parameters:
    -----------
    patient_id: int
        A unique id representing the patient in the process

    env: simpy.Environment
        The simulation environment

    args: Experiment
        Container class for the simulation parameters/results.
    """
    arrival_time = env.now

    with args.acute_ward.request() as acute_bed_request:
        yield acute_bed_request
        
        acute_admit_time = env.now
        wait_for_acute = acute_admit_time - arrival_time

        # used to avoid collecting stats from initial conditions...
        if collect_results:
            args.results['waiting_acute'].append(wait_for_acute)
        
        trace(f"{env.now:.2f}: Patient {patient_id} admitted to acute ward." \
              + f"(waited {wait_for_acute:.2f} days)")
        
        # Simulate acute care treatment
        acute_care_duration = args.acute_los.sample()
        yield env.timeout(acute_care_duration)
        
        trace(f"{env.now:.2f}: Patient {patient_id} discharged.")

## 4. Arrivals generator

This is a standard arrivals generator. We create stroke arrivals according to their distribution.

In [8]:
def stroke_arrivals_generator(env, args):
    """
    Arrival process for strokes.

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    args: Experiment
        The settings and input parameters for the simulation.
    """
    # use itertools as it provides an infinite loop
    # with a counter variable that we can use for unique Ids
    for patient_id in itertools.count(start=1):

        # the sample distribution is defined by the experiment.
        inter_arrival_time = args.arrival_strokes.sample()
        yield env.timeout(inter_arrival_time)

        args.results["n_arrivals"] += 1
        
        trace(f"{env.now:.2f}: Patient {patient_id}. Stroke arrival.")

        # patient enters pathway
        env.process(acute_stroke_pathway(patient_id, env, args))

In [9]:
def setup_initial_conditions(
    env: simpy.Environment, 
    args: Experiment
):
    """Set up initial conditions with patients already in the acute stroke queue

    Parameters:
    -----------
    env: simpy.Environment
       The simpy environment for the simulation

    args: Experiment
        The settings and input parameters for the simulation.        
    """
    # sample the no. patients to load into queue
    patients_to_load = args.init_conds.sample()
    trace(f"Patient to load: {patients_to_load}")

    # collect results or not?
    collect_results = args.init_cond_params["collect_results"]
    
    for initial_id in range(1, patients_to_load+1):
        # Create patients with negative IDs to distinguish them as init cond.
        # we may or may not want collect results for initial conditions
        env.process(acute_stroke_pathway(-initial_id, env, args, collect_results))
        trace(f"{env.now:.2f}: Patient {-initial_id} loaded into queue")

## 5. Single run function

In [10]:
def single_run(
    experiment, 
    rep=0,
    wu_period=WU_PERIOD,
    rc_period=RC_PERIOD
):
    """
    Perform a single run of the model and return the results

    Parameters:
    -----------

    experiment: Experiment
        The experiment/paramaters to use with model

    rep: int
        The replication number.

    wu_period: float, optional (default=WU_PERIOD)
        Warm-up period

    rc_period: float, optional (default=RC_PERIOD)
        The run length of the model
    """

    # reset all results variables to zero and empty
    experiment.init_results_variables()

    # set random number set to the replication no.
    # this controls sampling for the run.
    experiment.set_random_no_set(rep)

    # environment is (re)created inside single run
    env = simpy.Environment()

    # simpy resources
    experiment.acute_ward = simpy.Resource(env, experiment.n_acute_beds)

    trace("--- Pre-Simulation Actions ---")

    # schedule a warm_up period
    env.process(warmup_complete(wu_period, env, experiment))
    
    # load the acute stroke queue
    setup_initial_conditions(env, experiment)
    
    # we pass all arrival generators to simpy 
    env.process(stroke_arrivals_generator(env, experiment))

    trace("--- Start Simulation ---")
    
    # run model
    env.run(until=wu_period+rc_period)

    # quick stats
    results = {}
    results['mean_acute_wait'] = np.array(
        experiment.results["waiting_acute"]
    ).mean()

    # return single run results
    return results

In [ ]:
TRACE = True

# settings dictionary
init_cond_params = INIT_COND_PARAMS.copy()
init_cond_params["mode"] = "fixed"

# vary the fixed amount. Interpretation 10 = 1 in queue.
init_cond_params["fixed"] = 10

# vary if we collect results from initial condition stroke patients
init_cond_params["collect_results"] = False

# create experiment and pass in initial conditions
experiment = Experiment(init_cond_params=init_cond_params)

results = single_run(experiment, rep=1, wu_period=0.0, rc_period=10.0)
results

In [ ]:
experiment.results